# 42. Numbers in the paper

Computes the key numbers quoted in `NewGlenn_explosion_paper.tex` from the CSV products of notebooks 21–38 and
writes them as LaTeX macros to `tables/numbers.tex` (read by the paper with `\input{tables/numbers}`). Rerun
after notebooks 21–40, then recompile the paper. Each macro is printed with its value for checking.

In [1]:
import numpy as np
import pandas as pd
import newglenn_config as cfg

O = cfg.OUT_DIR
V = {}

def rng(a, b, d=0, sep='--'):
    lo, hi = f'{a:.{d}f}', f'{b:.{d}f}'
    return lo if lo == hi else f'{lo}{sep}{hi}'

## Origin time and weather (notebook 24)

In [2]:
fit = pd.read_csv(O / '24_origin_time' / 'origin_time_fit.csv')
best = fit[(fit.onset == 'onset_05') & (fit.model == 'weather + wind')].iloc[0]
t = pd.to_datetime(best.origin_time)
V['NGorigin'] = (t.round('100ms')).strftime('%H:%M:%S.%f')[:10]
V['NGoriginPrecise'] = (t.round('10ms')).strftime('%H:%M:%S.%f')[:11]
V['NGoriginSpread'] = f'{best.spread_s:.2f}'
wb = fit[fit.model != 'free c']
tt = pd.to_datetime(wb.origin_time)
V['NGoriginRange'] = tt.min().round('100ms').strftime('%H:%M:%S.%f')[:10] + '--' + tt.max().round('100ms').strftime('%S.%f')[:4]
free = fit[(fit.onset == 'onset_05') & (fit.model == 'free c')].iloc[0]
V['NGoriginFree'] = pd.to_datetime(free.origin_time).round('100ms').strftime('%H:%M:%S.%f')[:10]
V['NGspeedFree'] = f'{free.c_mps:.0f}'
w = pd.read_csv(O / '24_origin_time' / 'weather_at_origin.csv').set_index('station')
m = w.loc[cfg.EVENT.raw['weather']['stations']].mean()
from ksclib import weather
spd, frm = weather.wind_from_uv(m.u, m.v)
V['NGtemp'] = f'{m.temperature_c:.1f}'
V['NGrh'] = f'{m.rh:.0f}'
V['NGpressure'] = f'{m.pressure_pa / 100:.0f}'
V['NGwind'] = f'{float(spd):.1f}'
V['NGwindFrom'] = f'{float(frm):.0f}'
sp = pd.read_csv(O / '24_origin_time' / 'airwave_speeds.csv').set_index('station').loc[['B20', 'B14', 'B29', 'B05']]
V['NGczero'] = f'{sp.c0_mps.iloc[0]:.1f}'
V['NGceff'] = rng(sp.c_eff_mps.min(), sp.c_eff_mps.max(), 1)
V['NGheadwind'] = rng(-sp.wind_along_ray_mps.max(), -sp.wind_along_ray_mps.min(), 1)

## KSC weather towers (notebook 25)

In [3]:
to = pd.read_csv(O / '25_ksc_towers' / 'tower_origin.csv').set_index('model')
tp = pd.read_csv(O / '25_ksc_towers' / 'tower_path_speeds.csv')
tc = pd.read_csv(O / '25_ksc_towers' / 'tower_temperature_contrast.csv')
tm = pd.read_csv(O / '25_ksc_towers' / 'tower_means.csv')
low = to.loc[['towers, 6 ft T, 12 ft wind', 'towers, 54 ft']]
high = to.loc['towers, 204 ft']
V['NGtowerN'] = f"{tm[tm.lat.between(28.42, 28.66) & tm.lon.between(-80.70, -80.50)].station.nunique()}"
V['NGtowerShiftLow'] = f'{low.shift_vs_asos_s.abs().max():.1f}'
V['NGtowerShiftHigh'] = f'{abs(high.shift_vs_asos_s):.1f}'
V['NGtowerShiftHighDir'] = 'earlier' if high.shift_vs_asos_s < 0 else 'later'
V['NGtowerSpreadHigh'] = f'{high.spread_s:.2f}'
lowp = tp[tp.level != '204 ft']
V['NGtowerCpathLow'] = rng(lowp.c_path_mps.min(), lowp.c_path_mps.max(), 1)
V['NGtowerCpathHigh'] = rng(high.c_min, high.c_max, 1)
hp = tp[tp.level == '204 ft']
V['NGtowerHeadHigh'] = rng(-hp.wind_along_path_mps.max(), -hp.wind_along_path_mps.min(), 1)
lp = tp[tp.level == '6 ft T, 12 ft wind']
V['NGtowerHeadLow'] = rng(-lp.wind_along_path_mps.max(), -lp.wind_along_path_mps.min(), 1)
V['NGtowerTcoast'] = rng(tc.lo[0], tc.hi[0], 1)
V['NGtowerTinland'] = rng(tc.lo[1], tc.hi[1], 1)
V['NGtowerDTmax'] = f'{max(abs(tc.lo[2]), abs(tc.hi[2])):.1f}'

## Blast wave, decay and yield (notebook 34)

In [4]:
bp = pd.read_csv(O / '34_amplitude_vs_distance' / 'blast_parameters.csv')
b20 = bp[bp.station == 'B20'].iloc[0]
far = bp[bp.station != 'B20']
V['NGpBtwenty'] = f'{b20.P_pos_Pa:.0f}'
V['NGpFar'] = rng(far.P_pos_Pa.min(), far.P_pos_Pa.max())
V['NGtpos'] = rng(bp.t_pos_s.min(), bp.t_pos_s.max(), 2)
V['NGipos'] = rng(bp.I_pos_Pa_s.min(), bp.I_pos_Pa_s.max())
V['NGiposr'] = rng(np.round(bp.I_pos_x_r_Pa_s_km.min(), -1), np.round(bp.I_pos_x_r_Pa_s_km.max(), -1))
ratio = bp.P_pos_Pa / bp.P_neg_Pa.abs()
V['NGnegRatio'] = rng(np.round(ratio.min()), np.round(ratio.max()))
dec = pd.read_csv(O / '34_amplitude_vs_distance' / 'decay_fits.csv')
def n(metric, data='all', d=2):
    r = dec[(dec.metric == metric) & (dec.data == data)].iloc[0]
    return f'{r.n:.{d}f} \\pm {r.n_se:.{d}f}'
V['NGnPeak'] = n('P+ (Pa)')
V['NGnLow'] = n('peak 0.05-4 Hz (Pa)')
V['NGnImp'] = n('I+ (Pa s)', d=1)
V['NGnPeakCorr'] = f"{dec[(dec.metric == 'P+ (Pa)') & (dec.data == 'B20 peak corrected')].n.iloc[0]:.2f}"
de = pd.read_csv(O / '34_amplitude_vs_distance' / 'decimation_effect.csv')
loss = 100 * (1 - de.ratio_P)
b14 = loss[de.id.str.contains('B14')]
oth = loss[~de.id.str.contains('B14')]
V['NGdecBfourteen'] = rng(b14.min(), b14.max())
V['NGdecOther'] = rng(np.floor(oth.min()), np.ceil(oth.max()))
V['NGdecMax'] = f'{loss.max():.0f}'
y = pd.read_csv(O / '34_amplitude_vs_distance' / 'yield_estimates.csv')
V['NGyieldP'] = f'{y.W_from_P_surface_t.median():.0f}'
V['NGyieldPrange'] = rng(y.W_from_P_surface_t.min(), y.W_from_P_surface_t.max())
V['NGyieldT'] = f'{y.W_from_tpos_surface_t.median():.0f}'
V['NGyieldTrange'] = rng(y.W_from_tpos_surface_t.min(), y.W_from_tpos_surface_t.max())
V['NGzRange'] = rng(np.floor(y.Z_from_P.min() / 10) * 10, np.ceil(y.Z_from_P.max() / 10) * 10)
V['NGyieldVsSilber'] = f'{100 * abs(y.W_from_P_surface_t.median() / 131 - 1):.0f}'

### Blast-front advance and the video timeline

Near the pad the blast front is a shock travelling faster than sound, so the origin found by back-projecting
the airwave onsets at the speed of sound (notebook 24) is early. `physics.blast_front_time_advance` integrates
$(1 - 1/M)/c$ along the ray with the Kinney–Graham overpressure and the normal-shock Mach number, for the
free-air equivalent (twice the surface-burst TNT mass) of the yield from peak overpressure. A second estimate
uses the AMOS-6 model (`physics.mach_vs_overpressure`, Thompson et al. 2026): each station's measured $P_+$
extrapolated inward as $1/r$ to 10 m. Spherical spreading underestimates the near-source pressure (the
Kinney–Graham decay is steeper close in), so it gives a lower bound. The corrected
origin is compared with the video timeline of Silber & Scamfer (2026): first fireball at 00:59:59.9 UTC,
main flash at 01:00:03.6–04.4 UTC.

In [5]:
from ksclib.physics import blast_front_time_advance
d_m = bp.groupby('station').distance_km.first() * 1000
c_ray = sp.c_eff_mps.reindex(d_m.index)
W_med = 2 * y.W_from_P_surface_t.median() * 1000          # free-air kg
adv = pd.Series([blast_front_time_advance(d, W_med, c) for d, c in zip(d_m, c_ray)], index=d_m.index)
adv_lo = min(blast_front_time_advance(d, 2 * y.W_from_P_surface_t.min() * 1000, c) for d, c in zip(d_m, c_ray))
adv_hi = max(blast_front_time_advance(d, 2 * y.W_from_tpos_surface_t.max() * 1000, c) for d, c in zip(d_m, c_ray))
print(adv.round(3).to_dict(), round(adv_lo, 2), round(adv_hi, 2))
flash0 = pd.Timestamp('2026-05-29T01:00:03.6')
fire0 = pd.Timestamp('2026-05-29T00:59:59.9')
from ksclib.physics import mach_vs_overpressure
pst = bp.groupby('station').P_pos_Pa.median().loc[d_m.index]
def adv_sph(d, p, c):
    r = mach_vs_overpressure(speed_of_sound=c, reference_distance=d, reference_overpressure=p,
                             minimum_model_distance=10.0, spreading_exponent=1.0)
    return r['modeled_path_length_m'] / c - r['travel_time_s']
adv_s = pd.Series([adv_sph(d, p, c) for d, p, c in zip(d_m, pst, c_ray)], index=d_m.index)
print('spherical (AMOS-6) advance', adv_s.round(3).to_dict())
V['NGadvanceSph'] = rng(adv_s.min(), adv_s.max(), 1)
V['NGadvanceAll'] = rng(adv_s.min(), adv.max(), 1)
corr_lo = t + pd.Timedelta(seconds=float(adv_s.mean()))
corr = t + pd.Timedelta(seconds=float(adv.mean()))
V['NGoriginCorrRange'] = corr_lo.round('100ms').strftime('%H:%M:%S.%f')[:10] + '--' + corr.round('100ms').strftime('%S.%f')[:4]
V['NGflashGapRange'] = rng((flash0 - corr).total_seconds(), (flash0 - corr_lo).total_seconds(), 1)
V['NGadvance'] = rng(adv.min(), adv.max(), 1)
V['NGadvanceRange'] = rng(adv_lo, adv_hi, 1)
V['NGoriginCorr'] = corr.round('100ms').strftime('%H:%M:%S.%f')[:10]
V['NGflashGap'] = f'{(flash0 - corr).total_seconds():.1f}'
V['NGfireballGap'] = f'{(corr - fire0).total_seconds():.1f}'
V['NGflashFireballSep'] = f'{(flash0 - fire0).total_seconds():.1f}'
# pad position: launchpads sheet (newglenn.yaml) vs Silber & Scamfer (28.4706 N, 80.5400 W)
from obspy.geodetics import gps2dist_azimuth
stt = pd.read_csv(O / '21_station_map' / 'station_table.csv').set_index('station').loc[d_m.index]
d_ss = pd.Series([gps2dist_azimuth(28.4706, -80.5400, r.lat, r.lon)[0] for _, r in stt.iterrows()], index=stt.index)
pad_sep = gps2dist_azimuth(cfg.EVENT.site_lat, cfg.EVENT.site_lon, 28.4706, -80.5400)[0]
shift = ((d_m - d_ss) / c_ray).mean()       # origin moves later by this if the source is at the S&S point
print(f'pad separation {pad_sep:.0f} m; origin shift {shift:+.2f} s; per station', ((d_m - d_ss) / c_ray).round(2).to_dict())
V['NGpadSep'] = f'{np.round(pad_sep, -1):.0f}'
V['NGpadShift'] = f'{abs(shift):.1f}'
V['NGpadShiftDir'] = 'earlier' if shift < 0 else 'later'

{'B05': 0.671, 'B14': 0.664, 'B20': 0.63, 'B29': 0.67} 0.55 1.01
spherical (AMOS-6) advance {'B05': 0.437, 'B14': 0.395, 'B20': 0.335, 'B29': 0.489}
pad separation 216 m; origin shift -0.25 s; per station {'B05': -0.28, 'B14': -0.21, 'B20': -0.14, 'B29': -0.38}


## Coupling (notebook 28)

In [6]:
cm = pd.read_csv(O / '28_air_ground_coupling' / 'coupling_metrics.csv')
c14 = cm[cm.station == 'B14']
V['NGcompBfourteen'] = f'{np.median(c14.compliance_nm_Pa):.0f}'
V['NGcompBfourteenRange'] = f'${c14.compliance_nm_Pa.max():.0f}$ to ${c14.compliance_nm_Pa.min():.0f}$'
V['NGcompBtwentynine'] = f"{cm[cm.station == 'B29'].compliance_nm_Pa.iloc[0]:.0f}"
V['NGcompBzerofive'] = f"{cm[cm.station == 'B05'].compliance_nm_Pa.iloc[0]:.0f}"
neg = cm[cm.compliance_nm_Pa < 0].compliance_nm_Pa.abs()
V['NGcompRange'] = rng(neg.min(), neg.max())
ok = cm[cm.station != 'B20']
V['NGkRms'] = rng(ok.k_rms_um_s_Pa.min(), ok.k_rms_um_s_Pa.max(), 1)
V['NGkPeak'] = rng(ok.k_peak_um_s_Pa.min(), ok.k_peak_um_s_Pa.max(), 1)
a2 = pd.read_csv(O / '28_air_ground_coupling' / 'artemis2_comparison.csv')
art = a2[a2.source.str.startswith('Artemis')].set_index('station')
ngx = a2[a2.source.str.startswith('New')].set_index('station')
V['NGcompAtwoBzerofive'] = f"{art.loc['B05', 'compliance_nm_Pa']:.1f}"
V['NGcompAtwoBtwentynine'] = f"{art.loc['B29', 'compliance_nm_Pa']:.1f}"
V['NGcompAgree'] = f"{100 * (art.compliance_nm_Pa / ngx.compliance_nm_Pa - 1).abs().max():.0f}"
lr = pd.read_csv(O / '28_air_ground_coupling' / 'layer_resonance.csv')
lr = lr[~lr.pair.str.startswith('B20')]
V['NGfzero'] = rng(lr.f0_hz.min(), lr.f0_hz.max(), 1)

## Seismic waves and velocity model (notebooks 26 and 38)

In [7]:
sk = pd.read_csv(O / '26_seismic_arrivals' / 'seismic_picks.csv').set_index('station').loc[['B14', 'B29', 'B05']]
aic = sk[['aic_2_12_s', 'aic_1_5_s']]
V['NGfirstArrival'] = rng(aic.min().min(), aic.max().max(), 1)
d = sk.distance_m / 1000
vmin, vmax = (d / aic.max(axis=1)).min(), (d / aic.min(axis=1)).max()
V['NGvFirst'] = rng(np.floor(vmin * 10) / 10, np.ceil(vmax * 10) / 10, 1)
V['NGlead'] = rng(sk.lead_over_airwave_s.min(), sk.lead_over_airwave_s.max())
je = pd.read_csv(O / '38_horizontal_components' / 'joint_ensemble.csv')
bestfits = je.sort_values('misfit').groupby('fit').head(1)
V['NGvsTop'] = rng(np.floor(bestfits.vs1_m_s.min() / 10) * 10, np.ceil(bestfits.vs1_m_s.max() / 10) * 10)
V['NGtopThick'] = rng(np.floor(bestfits.h1_m.min() / 5) * 5, np.ceil(bestfits.h1_m.max() / 5) * 5)
V['NGhalfDepth'] = rng(np.floor(bestfits.depth_to_halfspace_m.min() / 5) * 5, np.ceil(bestfits.depth_to_halfspace_m.max() / 5) * 5)
V['NGvsHalf'] = f'{np.round(bestfits.vs4_m_s.mean() / 10) * 10:.0f}'

## Chaparral (notebook 32)

In [8]:
cw = pd.read_csv(O / '32_chaparral_sensitivity' / 'chaparral_windows.csv')
ev = cw.groupby(['epoch', 'source']).S_V_per_Pa.median().reset_index()
def ep(e):
    g = ev[ev.epoch == e].S_V_per_Pa
    return rng(g.min(), g.max(), 2)
V['NGchapEarly'] = ep('Jan-4 Apr (500 sps)')
V['NGchapLate'] = ep('14 Jun-Sep (500 sps)')
V['NGchapExplosion'] = f"{cw[cw.source.str.startswith('explosion')].S_V_per_Pa.median():.3f}"
V['NGchapWindows'] = f'{len(cw)}'
V['NGchapLaunches'] = f"{cw[cw.source.str.startswith('launch')].source.nunique()}"

## Network, channel status and instruments (notebooks 21, 22, 30)

In [9]:
WORDS = {1: 'one', 2: 'two', 3: 'three', 4: 'four', 5: 'five', 6: 'six', 7: 'seven', 8: 'eight', 9: 'nine'}
st = pd.read_csv(O / '21_station_map' / 'station_table.csv').set_index('station')
working = st[~st.status.str.startswith('none')]
V['NGdistRange'] = rng(st.distance_km.min(), st.distance_km.max(), 1)
V['NGdistRangeRound'] = rng(st.distance_km.min(), st.distance_km.max())
far = working.drop(index='B20')
V['NGfarRange'] = rng(far.distance_km.min(), far.distance_km.max())
V['NGnStations'] = WORDS[len(st)]
V['NGnStationsCap'] = WORDS[len(st)].capitalize()
V['NGnWorking'] = WORDS[len(working)]
for sta, name in (('B20', 'Btwenty'), ('B14', 'Bfourteen'), ('B29', 'Btwentynine'), ('B05', 'Bzerofive')):
    V['NGdist' + name] = f'{st.distance_km[sta]:.1f}'
cs = pd.read_csv(O / '22_data_overview' / 'channel_status.csv')
dead = cs[cs.status.str.startswith('no sensor') & cs.station.isin(['B01', 'B03', 'B24'])]
V['NGdeadDH'] = f"{dead[dead.channel.str[1] == 'H'].noise_counts.median():.0f}"
V['NGdeadDD'] = f"{dead[dead.channel.str[1] == 'D'].noise_counts.median():.0f}"
pol = pd.read_csv(O / '22_data_overview' / 'infrabsu_polarity.csv')
V['NGpolCorr'] = f'{pol.corr_B14_CD0.min():.2f}'
b = pd.read_csv(O / '30_spectral_ratios' / 'b14_summary.csv').iloc[0]
V['NGbRatio'] = f'{b.amp_ratio_CD2_CD1:.3f}'
V['NGbCal'] = f'{100 * (1 - b.amp_ratio_CD2_CD1):.0f}'
V['NGbCohHz'] = f'{np.round(b.coherence_098_up_to_hz, -1):.0f}'
V['NGresPeakDb'] = f'{b.resonance_peak_db:+.0f}'
V['NGresPeakHz'] = f'{np.round(b.resonance_peak_hz, -1):.0f}'
V['NGresMinDb'] = f'{b.resonance_min_db:.0f}'
V['NGresMinHz'] = f'{b.resonance_min_hz:.0f}'
V['NGnoiseDDzero'] = f'{b.noise_CD0_Pa:.1f}'
V['NGnoiseDDone'] = f'{b.noise_CD1_Pa:.1f}'
V['NGnoiseDDtwo'] = f'{b.noise_CD2_Pa:.2f}'
ci = pd.read_csv(O / '22_data_overview' / 'b20_chaparral_clip_intervals.csv')
sat = ci[ci.method.str.startswith('raw counts')]
V['NGclipN'] = f'{len(sat)}'
V['NGclipStart'] = f'{sat.start_s.min():.2f}'
V['NGclipEnd'] = f'{sat.end_s.max():.1f}'
cw2 = pd.read_csv(O / '32_chaparral_sensitivity' / 'chaparral_windows.csv')
cw2['t'] = pd.to_datetime(cw2.t, utc=True, format='mixed')
fault = cw2[cw2.epoch.str.startswith('4 Apr')]
per = fault.groupby('source').agg(t=('t', 'min'), S=('S_V_per_Pa', 'median'))
A = per[per.t < pd.Timestamp('2026-04-25', tz='UTC')].S
Bq = per[(per.t >= pd.Timestamp('2026-04-25', tz='UTC')) & (per.t < pd.Timestamp('2026-05-10', tz='UTC'))].S
Cq = per[per.t >= pd.Timestamp('2026-05-10', tz='UTC')].S
V['NGchapFaultA'] = f'{A.median():.3f}'
V['NGchapFaultB'] = f'{Bq.median():.2f}'
V['NGchapFaultC'] = rng(Cq.min(), Cq.max(), 2)
de2 = pd.read_csv(O / '34_amplitude_vs_distance' / 'decimation_effect.csv')
V['NGdecImpDur'] = f'{np.ceil(100 * max((de2.ratio_I - 1).abs().max(), (de2.ratio_t - 1).abs().max())):.0f}'

## Spectra (notebook 30) and remaining blast values (notebook 34)

In [10]:
sb = pd.read_csv(O / '30_spectral_ratios' / 'snr_bands.csv')
quiet = sb[sb.id.isin(['1R.B14.00.CD1', '1R.B14.00.CD2', '1R.B29.00.CD0'])]
V['NGsnrLow'] = f'{quiet.f_low_snr10.max():.1f}'
V['NGsnrHigh'] = f'{int(quiet.f_high_snr10.min() + 0.5)}'
pr = sb[sb.kind == 'pressure']
V['NGenergyLow'] = rng(pr['energy_0.5-5Hz_%'].min(), pr['energy_0.5-5Hz_%'].max())
V['NGpBtwentynine'] = f"{bp[bp.station == 'B29'].P_pos_Pa.iloc[0]:.0f}"
V['NGpBzerofive'] = f"{bp[bp.station == 'B05'].P_pos_Pa.iloc[0]:.0f}"
rs = pd.read_csv(O / '30_spectral_ratios' / 'reduced_pressure_spectra.csv')
pv = rs[rs.snr_ok].pivot_table(index='f_hz', columns='station', values='asd_reduced_Pa_per_rtHz')
db = 20 * np.log10(pv)
spread = (db.max(axis=1) - db.min(axis=1))[db.notna().sum(axis=1) >= 3]
V['NGredLow'] = f'{spread[spread.index <= 4].median():.0f}'
V['NGredHigh'] = f'{spread[(spread.index > 4) & (spread.index <= 45)].median():.0f}'

## Coupling details (notebook 28)

In [11]:
ok2 = cm[cm.station != 'B20']
V['NGlagMax'] = f'{ok2.compliance_lag_s.abs().max():.2f}'
V['NGcorrBfourteen'] = f"{cm[cm.station == 'B14'].compliance_r.median():.2f}"
V['NGcorrBtwentynine'] = f"{cm[cm.station == 'B29'].compliance_r.iloc[0]:.2f}"
V['NGcorrBzerofive'] = f"{cm[cm.station == 'B05'].compliance_r.iloc[0]:.2f}"
V['NGvpCorr'] = rng(ok2.v_on_p_r.abs().min(), ok2.v_on_p_r.abs().max(), 2)
V['NGkArtBzerofive'] = f"{art.loc['B05', 'k_rms_um_s_Pa']:.1f}"
V['NGkNgBzerofive'] = f"{ngx.loc['B05', 'k_rms_um_s_Pa']:.1f}"
V['NGkArtBtwentynine'] = f"{art.loc['B29', 'k_rms_um_s_Pa']:.1f}"
V['NGkNgBtwentynine'] = f"{ngx.loc['B29', 'k_rms_um_s_Pa']:.1f}"
lr2 = pd.read_csv(O / '28_air_ground_coupling' / 'layer_resonance.csv')
lr2['station'] = lr2.pair.str[:3]
c0 = lr2[lr2.station.isin(['B14', 'B29'])].C0_nm_Pa
V['NGczeroRange'] = rng(c0.min(), c0.max())
V['NGczeroBzerofive'] = f"{lr2[lr2.station == 'B05'].C0_nm_Pa.iloc[0]:.0f}"
for sta, name in (('B14', 'Bfourteen'), ('B29', 'Btwentynine'), ('B05', 'Bzerofive')):
    V['NGfzero' + name] = f"{lr2[lr2.station == sta].f0_hz.iloc[0]:.1f}"
b20c = cm[cm.station == 'B20'].iloc[0]
V['NGcompBtwenty'] = f'{b20c.compliance_nm_Pa:+.0f}'
V['NGcorrBtwenty'] = f'{b20c.compliance_r:+.2f}'

## Seismic arrivals, dispersion and models (notebooks 26, 36, 38)

In [12]:
sk2 = pd.read_csv(O / '26_seismic_arrivals' / 'seismic_picks.csv').set_index('station')
fs3 = sk2.loc[['B14', 'B29', 'B05']]
V['NGaicDiff'] = rng(fs3.first_arrival_spread_s.min(), fs3.first_arrival_spread_s.max(), 1)
V['NGwavePeak'] = rng(fs3.wavetrain_peak_s.min(), fs3.wavetrain_peak_s.max())
V['NGwaveSpeed'] = rng(fs3.v_peak_km_s.min(), fs3.v_peak_km_s.max(), 1)
V['NGpScatter'] = f'{fs3.first_arrival_s.max() - fs3.first_arrival_s.min():.1f}'
sa = pd.read_csv(O / '26_seismic_arrivals' / 'seismic_amplitudes.csv')
V['NGseisPeak'] = rng(sa.seismic_peak_um_s.min(), sa.seismic_peak_um_s.max())
V['NGairOverSeis'] = rng(sa.aircoupled_over_seismic.min(), sa.aircoupled_over_seismic.max())
b20s = sk2.loc['B20']
V['NGbtwentyNoise'] = rng(b20s.noise_rms_z_um_s / sk2.loc['B29', 'noise_rms_z_um_s'], b20s.noise_rms_z_um_s / sk2.loc['B05', 'noise_rms_z_um_s'])
V['NGbtwentyBurst'] = rng(np.floor(b20s.aic_2_12_s), np.ceil(b20s.wavetrain_peak_s))
V['NGbtwentyBurstSpeed'] = rng(b20s.distance_m / 1000 / b20s.wavetrain_peak_s, b20s.distance_m / 1000 / b20s.aic_2_12_s, 1)
dd = pd.read_csv(O / '36_velocity_model' / 'dispersion_data.csv').set_index('f_hz')
V['NGrayLowF'] = f'{dd.U.iloc[0]:.2f}'
mid = dd[(dd.index >= 1.5) & (dd.index <= 3.0)].U
V['NGrayMid'] = rng(mid.min(), mid.max(), 1)
ori = pd.read_csv(O / '38_horizontal_components' / 'horizontal_orientation.csv')
az = ori[(ori.window == 'Rayleigh 1-3 Hz') & ori.station.isin(['B14', 'B29'])].H1_azimuth_if_radial
az = (az + 180) % 360
V['NGdhAz'] = rng(az.min(), az.max())
zrt = pd.read_csv(O / '38_horizontal_components' / 'dispersion_ZRT.csv')
T = zrt[(zrt.comp == 'T') & zrt.station.isin(['B14', 'B29']) & zrt.ok]
t05 = T[T.f_hz == T.f_hz.min()].U_km_s
V['NGloveLowF'] = rng(t05.min(), t05.max(), 1)
jd = pd.read_csv(O / '38_horizontal_components' / 'joint_data.csv')
lv = jd[jd.kind == 'Love'].set_index('f_hz').value / 1000
V['NGloveMin'] = f'{lv.min():.2f}'
near = lv[lv <= lv.min() + 0.01].index
V['NGloveMinF'] = rng(near.min(), near.max(), 1)
hiT = T[(T.f_hz > 1.8) & (T.f_hz <= 3.4)].groupby('f_hz').U_km_s.median()
V['NGloveHigh'] = rng(hiT.min(), hiT.max(), 2)
amp = zrt[zrt.station.isin(['B14', 'B29']) & (zrt.f_hz <= 3.0)].pivot_table(index='f_hz', columns=['station', 'comp'], values='amp')
tr = [np.median(amp[(s_, 'T')] / amp[(s_, 'R')]) for s_ in ('B14', 'B29')]
V['NGtOverR'] = rng(min(tr), max(tr), 1)
hv = jd[jd.kind == 'H/V'].set_index('f_hz').value
V['NGhvLowF'] = f'{hv.iloc[0]:.0f}'
hvp = hv[(hv.index >= 1.4) & (hv.index <= 1.75)]
V['NGhvPeak'] = rng(np.floor(hvp.min()), np.ceil(hvp.max()))
V['NGhvPeakF'] = rng(hvp.index.min(), hvp.index.max(), 1)
bf = bestfits
V['NGlayerTwoThick'] = rng(np.floor(bf.h2_m.min() / 5) * 5, np.ceil(bf.h2_m.max() / 5) * 5)
V['NGlayerTwoVs'] = f'{np.round(bf.vs2_m_s.mean(), -1):.0f}'
V['NGlayerThreeThick'] = rng(np.floor(bf.h3_m.min() / 10) * 10, np.ceil(bf.h3_m.max() / 10) * 10)
V['NGlayerThreeVs'] = rng(np.round(bf.vs3_m_s.min(), -1), np.round(bf.vs3_m_s.max(), -1))
V['NGvpHalf'] = f'{max(1600.0, 1.9 * bf.vs4_m_s.mean()) / 1000:.1f}'
vm = pd.read_csv(O / '36_velocity_model' / 'best_model_B.csv', index_col=0)
V['NGvertVs'] = f'{vm.vs_m_s.iloc[1] / 1000:.1f}'
V['NGvertDepth'] = f'{np.round(vm.thickness_m.iloc[0] + vm.thickness_m.iloc[1], -1):.0f}'
V['NGnPeakRound'] = f"{dec[(dec.metric == 'P+ (Pa)') & (dec.data == 'all')].n.iloc[0]:.1f}"
V['NGvsHalfRound'] = f'{np.round(bf.vs4_m_s.mean(), -2):.0f}'
V['NGhalfDepthRound'] = rng(np.floor(bf.depth_to_halfspace_m.min() / 100) * 100, np.ceil(bf.depth_to_halfspace_m.max() / 100) * 100)
V['NGdepthMaxRound'] = f'{np.ceil(bf.depth_to_halfspace_m.max() / 100) * 100:.0f}'
hc = pd.read_csv(O / '38_horizontal_components' / 'horizontal_correlation_2026.csv')
b20corr = hc[hc.station == 'B20']['corr'].min()
V['NGbtwentyCorr'] = f'{np.floor(b20corr * 1e4) / 1e4:.4f}'

In [13]:
lines = ['% Generated by 42_paper_numbers.ipynb from the notebooks\' outputs; do not edit by hand.']
for k, v in V.items():
    lines.append(f'\\newcommand{{\\{k}}}{{{v}}}')
(cfg.PAPER_DIR / 'tables' / 'numbers.tex').write_text('\n'.join(lines) + '\n')
for k, v in V.items():
    print(f'{k:24s} {v}')

NGorigin                 01:00:02.4
NGoriginPrecise          01:00:02.37
NGoriginSpread           0.27
NGoriginRange            01:00:02.2--03.2
NGoriginFree             01:00:01.2
NGspeedFree              339
NGtemp                   26.4
NGrh                     87
NGpressure               1014
NGwind                   2.2
NGwindFrom               298
NGczero                  348.7
NGceff                   346.7--347.3
NGheadwind               1.5--2.0
NGtowerN                 19
NGtowerShiftLow          0.1
NGtowerShiftHigh         0.3
NGtowerShiftHighDir      earlier
NGtowerSpreadHigh        0.21
NGtowerCpathLow          346.4--347.5
NGtowerCpathHigh         344.7--345.8
NGtowerHeadHigh          2.6--3.5
NGtowerHeadLow           1.0--1.2
NGtowerTcoast            25.8--26.7
NGtowerTinland           24.9--26.5
NGtowerDTmax             0.8
NGpBtwenty               407
NGpFar                   264--342
NGtpos                   0.36--0.44
NGipos                   28--61
NGiposr         